# GitHub Repository Quality Predictor — Version 2
## Multi-Class Grading, Ensemble Theory & Bayesian Hyperparameter Optimization

This notebook implements end-to-end multi-class modeling for GitHub repository quality:
- **Target Classes**:
  - **Tier 0 (Low / Dormant)**: Score < 40
  - **Tier 1 (Emerging / Moderate)**: Score 40 – 65
  - **Tier 2 (Production-Grade)**: Score $\ge$ 65
- **Model Paradigms**:
  1. **Bagging Benchmark**: Multi-Class Random Forest (variance reduction via tree decorrelation)
  2. **Gradient Boosting**: LightGBM (bias reduction via histogram-based sequential residual fitting)
  3. **Bayesian Optimization**: Optuna (Tree-structured Parzen Estimator / TPE search over 5-fold CV)
  4. **Ensemble Blend**: Soft-voting probability aggregation combining bagging and boosting

### 1. Setup, Environment & Data Ingestion
Import libraries, configure visual themes, and load the 26-feature preprocessed Version 2 dataset (`data/processed/github_repos_v2.csv`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Modeling & Validation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import optuna

# Matplotlib visual settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Load preprocessed V2 dataset
data_path = Path('../data/processed/github_repos_v2.csv')
if not data_path.exists():
    data_path = Path('data/processed/github_repos_v2.csv')

df = pd.read_csv(data_path)
print(f"Dataset successfully loaded: {df.shape[0]:,} rows, {df.shape[1]} columns.")
df.head()

### 2. Feature Matrix ($X$) & Target Vector ($y$) Separation
Separate the 26 engineered features from the multi-class target (`quality_tier`) and inspect class proportions across the dataset.

In [ ]:
target_col = 'quality_tier'
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols]
y = df[target_col]

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape:   {y.shape}")

print("\n--- Target Class Distribution ---")
tier_counts = y.value_counts().sort_index()
tier_pcts = (y.value_counts(normalize=True).sort_index() * 100).round(2)

tier_labels = {
    0: 'Tier 0 (Low / Dormant)',
    1: 'Tier 1 (Emerging / Moderate)',
    2: 'Tier 2 (Production-Grade)'
}

dist_df = pd.DataFrame({
    'Tier Label': [tier_labels[i] for i in tier_counts.index],
    'Count': tier_counts.values,
    'Percentage (%)': tier_pcts.values
})
dist_df

### 3. Stratified 80/20 Train-Test Split
We hold out 20% of the dataset for untouched evaluation. Using `stratify=y` guarantees that both training and holdout sets retain identical class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} samples ({X_train.shape[0] / len(df) * 100:.1f}%)")
print(f"Test set:       {X_test.shape[0]:,} samples ({X_test.shape[0] / len(df) * 100:.1f}%)")
print(f"Features count: {X_train.shape[1]}")

# Stratification verification table
strat_check = pd.DataFrame({
    'Full Dataset %': (y.value_counts(normalize=True).sort_index() * 100).round(2),
    'Train Set %':    (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    'Test Set %':     (y_test.value_counts(normalize=True).sort_index() * 100).round(2)
})
strat_check

### 4. Random Forest: Bagging & Variance Reduction

#### **Theory of Bagging (Bootstrap Aggregation)**
An unconstrained decision tree exhibits **low bias** but **high variance** (overfitting to training noise).
Bagging reduces variance by training $B$ trees on independent bootstrap samples (sampling $N$ rows with replacement).
If $B$ estimators have pairwise correlation $\rho$ and variance $\sigma^2$, the variance of their ensemble average is:
$$\text{Var}(\bar{Z}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

#### **Why Random Forest Beats Standard Bagging**
In standard bagging, trees are strongly correlated ($\rho$ is high) because dominant features are chosen at the root of nearly every tree.
Random Forest breaks this correlation by restricting each split to a random subset of features:
$$m = \sqrt{p} \approx \sqrt{26} \approx 5 \text{ features}$$
This forces secondary features to be explored, driving $\rho \to 0$ and minimizing overall ensemble variance.

#### **Out-of-Bag (OOB) Validation**
For a bootstrap sample of size $N$ drawn with replacement:
$$P(\text{never selected}) = \left(1 - \frac{1}{N}\right)^N \xrightarrow{N \to \infty} \frac{1}{e} \approx 36.8\%$$
Roughly $36.8\%$ of training repositories are never seen by any given tree, serving as a built-in validation set (`oob_score=True`) without needing 5-fold cross-validation.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest Multi-Class Classifier...")
rf_model.fit(X_train, y_train)
print(f"Random Forest Training Complete! Out-of-Bag (OOB) Accuracy: {rf_model.oob_score_:.4f}")

# Predict discrete classes and continuous class probabilities
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)

### 5. Gradient Boosting: LightGBM Baseline

#### **Theory of Gradient Boosting (Optimization in Function Space)**
Unlike Bagging (which builds independent trees in parallel), Boosting builds trees **sequentially** to minimize a loss function $L(y, \hat{y})$ via gradient descent:
1. Tree 1 predicts initial probabilities $\hat{y}^{(1)}$.
2. Tree 2 is fitted directly to the **negative gradients (pseudo-residuals)** of Tree 1:
   $$g_i = -\left[ \frac{\partial L(y_i, \hat{y}_i)}{\partial \hat{y}_i} \right]$$
3. Each subsequent tree scales its step size by the **learning rate (shrinkage)** $\eta \in (0, 1]$:
   $$\hat{y}^{(m)} = \hat{y}^{(m-1)} + \eta \cdot f_m(x)$$

#### **LightGBM vs. XGBoost Architecture**
- **Histogram Binning**: Continuous floats are mapped into 256 discrete integer bins (`uint8_t`), reducing memory by $80\%$ and speeding up split discovery by $15\times$.
- **Leaf-wise (Best-first) Growth**: Splits the single leaf that maximizes global loss reduction rather than splitting level-by-level, yielding lower loss for equivalent leaf counts.

In [ ]:
lgb_base = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

print("Training Default LightGBM Baseline...")
lgb_base.fit(X_train, y_train)
print("Default LightGBM training complete!")

y_pred_lgb_base = lgb_base.predict(X_test)
y_proba_lgb_base = lgb_base.predict_proba(X_test)

### 6. Bayesian Hyperparameter Optimization with Optuna

#### **Why Bayesian Optimization Over Grid / Random Search?**
- **Grid Search** evaluates combinations blindly across unpromising regions ($4^5 = 1,024$ fits).
- **Random Search** samples uniformly, treating every region with equal likelihood.
- **Optuna's Tree-structured Parzen Estimator (TPE)** models the probability distribution of hyperparameters given past trial performance:
  $$P(\theta \mid y) = \begin{cases} \ell(\theta) & \text{if } y > y^* \quad (\text{top 15\% trials}) \\ g(\theta) & \text{if } y \le y^* \quad (\text{remaining 85\% trials}) \end{cases}$$
  It samples the next parameter candidate by maximizing the **Expected Improvement** ratio $\frac{\ell(\theta)}{g(\theta)}$, concentrating compute in the highest-performing regions.

In [ ]:
def objective(trial):
    # Hyperparameter search space
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
        
        # Tunable parameters:
        'n_estimators': trial.suggest_int('n_estimators', 80, 250),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }

    # 5-fold Stratified CV on training set (optimizing Macro F1)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = lgb.LGBMClassifier(**params)
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    
    return scores.mean()

print("Launching Optuna Bayesian Optimization (30 trials across 5 stratified folds)...\n")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nOptimization Complete! Best CV Macro F1: {study.best_value:.4f}")
print("Discovered Optimal Hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k:18s}: {v}")

### 7. Training the Tuned LightGBM Model
We now train a final LightGBM model across the entire `X_train` dataset using the hyperparameters discovered by Optuna.

In [ ]:
best_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    **study.best_params
}

tuned_lgb = lgb.LGBMClassifier(**best_params)
print("Training Tuned LightGBM model on full training set...")
tuned_lgb.fit(X_train, y_train)
print("Tuned LightGBM training complete!")

y_pred_tuned_lgb = tuned_lgb.predict(X_test)
y_proba_tuned_lgb = tuned_lgb.predict_proba(X_test)

### 8. 5-Fold Stratified Cross-Validation Verification
Comparing the cross-fold stability (Mean $\pm$ Std) of Random Forest, Default LightGBM, and Tuned LightGBM across 5 folds.

In [ ]:
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'acc': 'accuracy', 'f1_macro': 'f1_macro', 'roc_auc': 'roc_auc_ovr'}

print("Evaluating 5-fold cross-validation across all models...")
rf_cv = cross_validate(rf_model, X_train, y_train, cv=cv_splitter, scoring=scoring, n_jobs=-1)
lgb_base_cv = cross_validate(lgb_base, X_train, y_train, cv=cv_splitter, scoring=scoring, n_jobs=-1)
lgb_tuned_cv = cross_validate(tuned_lgb, X_train, y_train, cv=cv_splitter, scoring=scoring, n_jobs=-1)

def format_cv(name, res):
    return {
        'Model': name,
        'CV Accuracy': f"{res['test_acc'].mean():.4f} ± {res['test_acc'].std():.4f}",
        'CV Macro F1': f"{res['test_f1_macro'].mean():.4f} ± {res['test_f1_macro'].std():.4f}",
        'CV Macro ROC-AUC': f"{res['test_roc_auc'].mean():.4f} ± {res['test_roc_auc'].std():.4f}"
    }

cv_table = pd.DataFrame([
    format_cv('Random Forest (v2)', rf_cv),
    format_cv('Default LightGBM (v2)', lgb_base_cv),
    format_cv('Tuned LightGBM (v2)', lgb_tuned_cv)
])
display(cv_table)

### 9. Comprehensive Evaluation on Holdout Test Set
Calculating final holdout metrics (Accuracy, Macro Precision, Macro Recall, Macro F1, One-vs-Rest ROC-AUC).

In [ ]:
def compute_metrics(model_name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Macro Precision': round(p, 4),
        'Macro Recall': round(r, 4),
        'Macro F1': round(f1, 4),
        'Macro ROC-AUC (OvR)': round(auc, 4)
    }

results = [
    compute_metrics('Random Forest (v2)', y_test, y_pred_rf, y_proba_rf),
    compute_metrics('Default LightGBM (v2)', y_test, y_pred_lgb_base, y_proba_lgb_base),
    compute_metrics('Tuned LightGBM (v2)', y_test, y_pred_tuned_lgb, y_proba_tuned_lgb)
]

display(pd.DataFrame(results))

target_names = ['Tier 0 (Low)', 'Tier 1 (Emerging)', 'Tier 2 (Production)']
print("\n" + "="*35 + " TUNED LIGHTGBM REPORT " + "="*35)
print(classification_report(y_test, y_pred_tuned_lgb, target_names=target_names, digits=4))

### 10. Normalized 3×3 Confusion Matrices
Visualizing misclassifications across tier boundaries for Random Forest vs. Tuned LightGBM.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
tier_labels_short = ['Tier 0\n(Low)', 'Tier 1\n(Emerging)', 'Tier 2\n(Production)']

# Random Forest Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf, normalize='true')
sns.heatmap(
    cm_rf, annot=True, fmt='.2%', cmap='Blues',
    xticklabels=tier_labels_short, yticklabels=tier_labels_short,
    ax=axes[0], cbar=False
)
axes[0].set_title('Random Forest (Normalized)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Tier')
axes[0].set_ylabel('True Tier')

# Tuned LightGBM Matrix
cm_lgb = confusion_matrix(y_test, y_pred_tuned_lgb, normalize='true')
sns.heatmap(
    cm_lgb, annot=True, fmt='.2%', cmap='Greens',
    xticklabels=tier_labels_short, yticklabels=tier_labels_short,
    ax=axes[1], cbar=False
)
axes[1].set_title('Tuned LightGBM (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Tier')
axes[1].set_ylabel('True Tier')

plt.tight_layout()
plt.show()

### 11. Soft-Voting Ensemble (Random Forest + Tuned LightGBM)
Combines the predicted class probability distributions from Bagging and Boosting:
$$P_{\text{ensemble}}(\text{Tier}_k) = \frac{P_{\text{RF}}(\text{Tier}_k) + P_{\text{Tuned LGB}}(\text{Tier}_k)}{2}$$

In [ ]:
y_proba_ensemble = (y_proba_rf + y_proba_tuned_lgb) / 2.0
y_pred_ensemble = np.argmax(y_proba_ensemble, axis=1)

metrics_ensemble = compute_metrics('Ensemble (RF + Tuned LGB Blend)', y_test, y_pred_ensemble, y_proba_ensemble)
results.append(metrics_ensemble)

print("--- Final Holdout Test Performance Comparison ---")
display(pd.DataFrame(results))

print("\n" + "="*35 + " ENSEMBLE REPORT " + "="*35)
print(classification_report(y_test, y_pred_ensemble, target_names=target_names, digits=4))

### 12. Feature Importance Comparison
Compares the top 15 features across Random Forest (Mean Decrease in Impurity / Gini) and Tuned LightGBM (Split Frequency).

In [ ]:
rf_imp = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
lgb_imp = pd.Series(tuned_lgb.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

rf_imp.sort_values().plot(kind='barh', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Top 15 Features — Random Forest (Gini Importance)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Importance Score')

lgb_imp.sort_values().plot(kind='barh', ax=axes[1], color='#1f774d')
axes[1].set_title('Top 15 Features — Tuned LightGBM (Split Count Importance)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

### 13. Model Artifact Serialization
Save trained Version 2 models to `models/` directory using `joblib`.

In [ ]:
models_dir = Path('../models')
if not models_dir.exists():
    models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)

rf_v2_path = models_dir / 'random_forest_v2.pkl'
lgb_v2_path = models_dir / 'lightgbm_v2.pkl'

joblib.dump(rf_model, rf_v2_path)
joblib.dump(tuned_lgb, lgb_v2_path)

print(f"Saved Random Forest v2 model to: {rf_v2_path.resolve()}")
print(f"Saved Tuned LightGBM v2 model to: {lgb_v2_path.resolve()}")
print(f"RF file size:  {rf_v2_path.stat().st_size / (1024 * 1024):.2f} MB")
print(f"LGB file size: {lgb_v2_path.stat().st_size / (1024 * 1024):.2f} MB")